In [2]:
%%writefile failsafe_agent.py
# Your Python code goes here
def hello():
    return "Hello from failsafe_agent!"
# failsafe_agent.py
import os, time, uuid, unicodedata, re, ast, difflib
from collections import Counter
from math import sqrt

# -----------------------------
# CONFIG
# -----------------------------
ALLOWED_MODELS = {"gpt-4o-mini", "gpt-4o"}
FALLBACK_MODEL = "gpt-4o-mini"
MAX_PROMPT_CHARS = 8000
MAX_OUTPUT_CHARS = 4000
SUPPORTED_EXTS = {".pdf", ".txt", ".md", ".docx"}
MAX_FILE_MB = 25

# -----------------------------
# HELPERS
# -----------------------------
def _normalize(text: str) -> str:
    return unicodedata.normalize("NFKC", text).strip()

def _truncate(s: str, limit: int) -> tuple[str, bool]:
    return (s[:limit], len(s) > limit)

def _extract_code_block(query: str) -> str | None:
    start = query.find("```")
    if start == -1: return None
    end = query.find("```", start + 3)
    if end == -1: return None
    return query[start + 3:end].strip()

# -----------------------------
# INPUT SAFETY
# -----------------------------
SENSITIVE_CATEGORIES = {
    "self_harm": [r"\bsuicide\b", r"\bkill myself\b", r"\bself harm\b"],
    "violence": [r"\bkill\b", r"\bmake a bomb\b", r"\battack\b"],
    "illegal": [r"\bhack\b", r"\bcredit card\b", r"\bwifi password\b", r"\bcrack\b"],
    "personal_data": [r"\bssn\b", r"\baadhaar\b", r"\bpan\b", r"\bupi\b", r"\bpassword\b"],
}

def is_query_allowed(text: str, max_len: int = 5000) -> tuple[bool, str]:
    if not text or len(text.strip()) < 3:
        return False, "Empty/too short query."
    if len(text) > max_len:
        return False, f"Query too long ({len(text)} chars). Limit: {max_len}."
    lower = text.lower()
    for cat, patterns in SENSITIVE_CATEGORIES.items():
        for p in patterns:
            if re.search(p, lower):
                return False, f"Blocked sensitive content: {cat}"
    return True, "OK"

# -----------------------------
# MALICIOUS CODE CHECK
# -----------------------------
FORBIDDEN_CALL_NAMES = {"eval","exec","compile","os.system","subprocess.Popen","subprocess.call","subprocess.run"}

def validate_user_query(query: str, code_snippet: str | None = None) -> tuple[bool,str]:
    if code_snippet:
        try:
            tree = ast.parse(code_snippet)
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_CALL_NAMES:
                        return False, f"Forbidden call: {node.func.id}"
        except SyntaxError:
            return False, "Syntax error in code block."
    return True, "OK"

# -----------------------------
# PLAGIARISM CHECK
# -----------------------------
def _normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return s.strip()

def cosine_token(a: str, b: str) -> float:
    A = Counter(_normalize_text(a).split())
    B = Counter(_normalize_text(b).split())
    dot = sum(A[t]*B[t] for t in set(A)|set(B))
    magA = sqrt(sum(v*v for v in A.values()))
    magB = sqrt(sum(v*v for v in B.values()))
    return dot/(magA*magB+1e-9)

def check_plagiarism(generated: str, corpus: list[str], cosine_thresh: float=0.85) -> dict:
    best = {"source_idx":None,"cosine":0.0}
    for i,ref in enumerate(corpus):
        cos = cosine_token(generated,ref)
        if cos>best["cosine"]:
            best={"source_idx":i,"cosine":cos}
    flagged = best["cosine"]>=cosine_thresh
    return {"flagged":flagged,"best_match":best}

# -----------------------------
# LLM WRAPPERS (dummy for demo)
# -----------------------------
class DummyLLM:
    def __init__(self,name,key): self.name=name; self.key=key
    def invoke(self,prompt): return {"content":f"Echo from {self.name}: {prompt[:200]}"}

def get_llm(model_name: str, api_key_env: str="LLM_API_KEY"):
    api_key=os.getenv(api_key_env,"dummy_key")
    if model_name not in ALLOWED_MODELS:
        raise ValueError(f"Unsupported model {model_name}")
    return DummyLLM(model_name,api_key)

def safe_invoke(llm,prompt:str)->dict:
    resp=llm.invoke(prompt)
    if not resp or "content" not in resp: raise ValueError("Empty response")
    return resp

# -----------------------------
# PIPELINE
# -----------------------------
def answer_pipeline(user_query: str, reference_corpus: list[str], model_name: str="gpt-4o-mini"):
    trace_id=str(uuid.uuid4())[:8]
    user_query=_normalize(user_query)

    # 1) Input restriction
    ok,reason=is_query_allowed(user_query)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 2) Init LLM
    try: llm=get_llm(model_name)
    except Exception as e: return {"status":"config_error","reason":str(e),"trace":trace_id}

    # 3) Malicious code check
    code_block=_extract_code_block(user_query)
    ok,reason=validate_user_query(user_query,code_block)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 4) Invoke
    try: resp=safe_invoke(llm,user_query)
    except Exception as e: return {"status":"llm_error","reason":str(e),"trace":trace_id}

    content, trunc=_truncate(resp["content"],MAX_OUTPUT_CHARS)

    # 5) Plagiarism
    check=check_plagiarism(content,reference_corpus)
    if check["flagged"]:
        return {"status":"plagiarism_flagged","details":check,"llm_preview":content[:200],"trace":trace_id}

    return {"status":"ok","answer":content,"trace":trace_id}

Writing failsafe_agent.py


In [4]:
result = answer_pipeline("Explain quantum computing basics", REFERENCE_CORPUS)
if result["status"] == "ok":
    print("✅ Safe answer:\n")
    print(result["answer"])
else:
    print("⚠️ Blocked:", result["reason"])

✅ Safe answer:

Echo from gpt-4o-mini: Explain quantum computing basics


In [5]:
%%writefile app_gradio.py
import gradio as gr
from failsafe_agent import answer_pipeline

REFERENCE_CORPUS = [
    "This is an example paper text ...",
    "Another doc with content ..."
]

def run_agent(user_query):
    result = answer_pipeline(user_query, REFERENCE_CORPUS, model_name="gpt-4o-mini")
    # Format output nicely
    if result.get("status") == "ok":
        return f"✅ Safe Answer:\n\n{result['answer']}"
    elif result.get("status") == "plagiarism_flagged":
        return f"⚠️ Plagiarism flagged\n\nPreview:\n{result['llm_preview']}"
    else:
        return f"⚠️ {result.get('status')}: {result.get('reason')}"

demo = gr.Interface(
    fn=run_agent,
    inputs=gr.Textbox(lines=4, placeholder="Enter your query here..."),
    outputs="text",
    title="🛡️ Failsafe Agent Demo",
    description="Shows safety checks (blocked queries, malicious code, plagiarism) before invoking the LLM."
)

if __name__ == "__main__":
    demo.launch()

Writing app_gradio.py


In [6]:
!pip install gradio --quiet
!python app_gradio.py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 11.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.0 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://7a46b810f8543ff255.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Fac

In [9]:
%%writefile app_mcp.py
import os, uuid, re, ast, unicodedata
from collections import Counter
from math import sqrt
import gradio as gr

# -----------------------------
# CONFIG
# -----------------------------
ALLOWED_MODELS = {"gpt-4o-mini", "gpt-4o"}
MAX_OUTPUT_CHARS = 4000

# -----------------------------
# HELPERS
# -----------------------------
def _normalize(text: str) -> str:
    return unicodedata.normalize("NFKC", text).strip()

def _truncate(s: str, limit: int) -> tuple[str, bool]:
    return (s[:limit], len(s) > limit)

def _extract_code_block(query: str) -> str | None:
    start = query.find("```")
    if start == -1: return None
    end = query.find("```", start + 3)
    if end == -1: return None
    return query[start + 3:end].strip()

# -----------------------------
# INPUT SAFETY
# -----------------------------
SENSITIVE_CATEGORIES = {
    "self_harm": [r"\bsuicide\b", r"\bkill myself\b", r"\bself harm\b"],
    "violence": [r"\bkill\b", r"\bmake a bomb\b", r"\battack\b"],
    "illegal": [r"\bhack\b", r"\bcredit card\b", r"\bwifi password\b", r"\bcrack\b"],
    "personal_data": [r"\bssn\b", r"\baadhaar\b", r"\bpan\b", r"\bupi\b", r"\bpassword\b"],
}

def is_query_allowed(text: str, max_len: int = 5000) -> tuple[bool, str]:
    if not text or len(text.strip()) < 3:
        return False, "Empty/too short query."
    if len(text) > max_len:
        return False, f"Query too long ({len(text)} chars). Limit: {max_len}."
    lower = text.lower()
    for cat, patterns in SENSITIVE_CATEGORIES.items():
        for p in patterns:
            if re.search(p, lower):
                return False, f"Blocked sensitive content: {cat}"
    return True, "OK"

# -----------------------------
# MALICIOUS CODE CHECK
# -----------------------------
# -----------------------------
# MALICIOUS CODE CHECK
# -----------------------------
import re, ast

DANGEROUS_PATTERNS = [
    r"\bos\.system\s*\(",
    r"\bsubprocess\.(Popen|run|call)\s*\(",
    r"\beval\s*\(",
    r"\bexec\s*\(",
    r"\b__import__\s*\(",
    r"\brm\s+-rf\b",
    r"\bsudo\b",
    r"\bchmod\b",
    r"\bchown\b",
    r"\bwget\b",
    r"\bcurl\b",
    r"\bnc\b",
    r"\bscp\b",
]

FORBIDDEN_CALL_NAMES = {"eval","exec","compile","os.system","subprocess.Popen","subprocess.call","subprocess.run"}

def contains_dangerous_strings(s: str) -> list[str]:
    s_lower = s.lower()
    return [pat for pat in DANGEROUS_PATTERNS if re.search(pat, s_lower)]

def validate_user_query(query: str, code_snippet: str | None = None) -> tuple[bool, str]:
    # 1) Raw string heuristics
    hits = contains_dangerous_strings(query)
    if hits:
        return False, f"Blocked due to dangerous patterns: {hits}"

    # 2) AST analysis if fenced code exists
    if code_snippet:
        try:
            tree = ast.parse(code_snippet)
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_CALL_NAMES:
                        return False, f"Forbidden call: {node.func.id}"
                    if isinstance(node.func, ast.Attribute):
                        parts = []
                        cur = node.func
                        while isinstance(cur, ast.Attribute):
                            parts.append(cur.attr)
                            cur = cur.value
                        if hasattr(cur, "id"):
                            parts.append(cur.id)
                        parts.reverse()
                        name = ".".join(parts)
                        if name in FORBIDDEN_CALL_NAMES:
                            return False, f"Forbidden call: {name}"
        except SyntaxError:
            return False, "Syntax error in code block."
    return True, "OK"

# -----------------------------
# PLAGIARISM CHECK
# -----------------------------
def _normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return s.strip()

def cosine_token(a: str, b: str) -> float:
    A = Counter(_normalize_text(a).split())
    B = Counter(_normalize_text(b).split())
    dot = sum(A[t]*B[t] for t in set(A)|set(B))
    magA = sqrt(sum(v*v for v in A.values()))
    magB = sqrt(sum(v*v for v in B.values()))
    return dot/(magA*magB+1e-9)

def check_plagiarism(generated: str, corpus: list[str], cosine_thresh: float=0.85) -> dict:
    best = {"source_idx":None,"cosine":0.0}
    for i,ref in enumerate(corpus):
        cos = cosine_token(generated,ref)
        if cos>best["cosine"]:
            best={"source_idx":i,"cosine":cos}
    flagged = best["cosine"]>=cosine_thresh
    return {"flagged":flagged,"best_match":best}

# -----------------------------
# LLM WRAPPER (dummy for demo)
# -----------------------------
class DummyLLM:
    def __init__(self,name,key): self.name=name; self.key=key
    def invoke(self,prompt): return {"content":f"Echo from {self.name}: {prompt[:200]}"}

def get_llm(model_name: str):
    if model_name not in ALLOWED_MODELS:
        raise ValueError(f"Unsupported model {model_name}")
    return DummyLLM(model_name,"dummy_key")

def safe_invoke(llm,prompt:str)->dict:
    resp=llm.invoke(prompt)
    if not resp or "content" not in resp: raise ValueError("Empty response")
    return resp

# -----------------------------
# PIPELINE
# -----------------------------
def answer_pipeline(user_query: str, reference_corpus: list[str], model_name: str="gpt-4o-mini"):
    trace_id=str(uuid.uuid4())[:8]
    user_query=_normalize(user_query)

    # 1) Input restriction
    ok,reason=is_query_allowed(user_query)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 2) Init LLM
    try: llm=get_llm(model_name)
    except Exception as e: return {"status":"config_error","reason":str(e),"trace":trace_id}

    # 3) Malicious code check
    code_block=_extract_code_block(user_query)
    ok,reason=validate_user_query(user_query,code_block)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 4) Invoke
    try: resp=safe_invoke(llm,user_query)
    except Exception as e: return {"status":"llm_error","reason":str(e),"trace":trace_id}

    content, trunc=_truncate(resp["content"],MAX_OUTPUT_CHARS)

    # 5) Plagiarism
    check=check_plagiarism(content,reference_corpus)
    if check["flagged"]:
        return {"status":"plagiarism_flagged","details":check,"llm_preview":content[:200],"trace":trace_id}

    return {"status":"ok","answer":content,"trace":trace_id}

# -----------------------------
# GRADIO DEMO
# -----------------------------
REFERENCE_CORPUS = [
    "This is an example paper text ...",
    "Another doc with content ..."
]

def run_agent(user_query):
    result = answer_pipeline(user_query, REFERENCE_CORPUS, model_name="gpt-4o-mini")
    if result.get("status") == "ok":
        return f"✅ Safe Answer:\n\n{result['answer']}"
    elif result.get("status") == "plagiarism_flagged":
        return f"⚠️ Plagiarism flagged\n\nPreview:\n{result['llm_preview']}"
    else:
        return f"⚠️ {result.get('status')}: {result.get('reason')}"

demo = gr.Interface(
    fn=run_agent,
    inputs=gr.Textbox(lines=4, placeholder="Enter your query here..."),
    outputs="text",
    title="🛡️ Failsafe Agent MCP",
    description="Demonstrates real-time safety checks before invoking the LLM."
)

if __name__ == "__main__":
    demo.launch()


Overwriting app_mcp.py


In [10]:
!pip install gradio --quiet
!python app_mcp.py

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://be8bfca61360d4e6f0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
^C
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://be8bfca61360d4e6f0.gradio.live


In [11]:
%%writefile app_mcp.py

UsageError: %%writefile is a cell magic, but the cell body is empty.


In [12]:
%%writefile app_mcp.py

UsageError: %%writefile is a cell magic, but the cell body is empty.


In [14]:
%%writefile app_mcp.py
import os, uuid, re, ast, unicodedata
from collections import Counter
from math import sqrt

# -----------------------------
# CONFIG
# -----------------------------
ALLOWED_MODELS = {"gpt-4o-mini", "gpt-4o"}
MAX_OUTPUT_CHARS = 4000

# -----------------------------
# HELPERS
# -----------------------------
def _normalize(text: str) -> str:
    return unicodedata.normalize("NFKC", text).strip()

def _truncate(s: str, limit: int) -> tuple[str, bool]:
    return (s[:limit], len(s) > limit)

def _extract_code_block(query: str) -> str | None:
    start = query.find("```")
    if start == -1: return None
    end = query.find("```", start + 3)
    if end == -1: return None
    return query[start + 3:end].strip()

# -----------------------------
# INPUT SAFETY (hardcoded)
# -----------------------------
SENSITIVE_CATEGORIES = {
    "self_harm": [r"\bsuicide\b", r"\bkill myself\b", r"\bself harm\b"],
    "violence": [r"\bkill\b", r"\bmake a bomb\b", r"\battack\b"],
    "illegal": [r"\bhack\b", r"\bcredit card\b", r"\bwifi password\b", r"\bcrack\b"],
    "personal_data": [r"\bssn\b", r"\baadhaar\b", r"\bpan\b", r"\bupi\b", r"\bpassword\b"],
}

def is_query_allowed(text: str, max_len: int = 5000) -> tuple[bool, str]:
    if not text or len(text.strip()) < 3:
        return False, "Empty/too short query."
    if len(text) > max_len:
        return False, f"Query too long ({len(text)} chars). Limit: {max_len}."
    lower = text.lower()
    for cat, patterns in SENSITIVE_CATEGORIES.items():
        for p in patterns:
            if re.search(p, lower):
                return False, f"Blocked sensitive content: {cat}"
    return True, "OK"

# -----------------------------
# MALICIOUS CODE CHECK (hardcoded)
# -----------------------------
DANGEROUS_PATTERNS = [
    r"\bos\.system\s*\(",
    r"\bsubprocess\.(Popen|run|call)\s*\(",
    r"\beval\s*\(",
    r"\bexec\s*\(",
    r"\b__import__\s*\(",
    r"\brm\s+-rf\b",
]

FORBIDDEN_CALL_NAMES = {"eval","exec","compile","os.system","subprocess.Popen","subprocess.call","subprocess.run"}

def contains_dangerous_strings(s: str) -> list[str]:
    s_lower = s.lower()
    return [pat for pat in DANGEROUS_PATTERNS if re.search(pat, s_lower)]

def validate_user_query(query: str, code_snippet: str | None = None) -> tuple[bool, str]:
    hits = contains_dangerous_strings(query)
    if hits:
        return False, f"Blocked due to dangerous patterns: {hits}"
    if code_snippet:
        try:
            tree = ast.parse(code_snippet)
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_CALL_NAMES:
                        return False, f"Forbidden call: {node.func.id}"
        except SyntaxError:
            return False, "Syntax error in code block."
    return True, "OK"

# -----------------------------
# PLAGIARISM CHECK
# -----------------------------
def _normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return s.strip()

def cosine_token(a: str, b: str) -> float:
    A = Counter(_normalize_text(a).split())
    B = Counter(_normalize_text(b).split())
    dot = sum(A[t]*B[t] for t in set(A)|set(B))
    magA = sqrt(sum(v*v for v in A.values()))
    magB = sqrt(sum(v*v for v in B.values()))
    return dot/(magA*magB+1e-9)

def check_plagiarism(generated: str, corpus: list[str], cosine_thresh: float=0.85) -> dict:
    best = {"source_idx":None,"cosine":0.0}
    for i,ref in enumerate(corpus):
        cos = cosine_token(generated,ref)
        if cos>best["cosine"]:
            best={"source_idx":i,"cosine":cos}
    flagged = best["cosine"]>=cosine_thresh
    return {"flagged":flagged,"best_match":best}

# -----------------------------
# LLM WRAPPER (dummy for demo)
# -----------------------------
class DummyLLM:
    def __init__(self,name,key): self.name=name; self.key=key
    def invoke(self,prompt): return {"content":f"Echo from {self.name}: {prompt[:200]}"}

def get_llm(model_name: str):
    if model_name not in ALLOWED_MODELS:
        raise ValueError(f"Unsupported model {model_name}")
    return DummyLLM(model_name,"dummy_key")

def safe_invoke(llm,prompt:str)->dict:
    resp=llm.invoke(prompt)
    if not resp or "content" not in resp: raise ValueError("Empty response")
    return resp

# -----------------------------
# LLM SAFETY FILTER (intelligent)
# -----------------------------
def llm_safety_filter(llm, query: str) -> bool:
    """
    Ask the LLM to classify query safety.
    Returns True if safe, False if unsafe.
    """
    prompt = f"""
    You are a safety filter. Classify the following query:
    "{query}"
    Respond with only one word: SAFE or UNSAFE.
    """
    resp = llm.invoke(prompt)
    label = resp["content"].strip().upper()
    return label == "SAFE"

def validate_with_llm(llm, query: str) -> tuple[bool, str]:
    if not llm_safety_filter(llm, query):
        return False, "Blocked by LLM safety filter"
    return True, "OK"

# -----------------------------
# HYBRID PIPELINE
# -----------------------------
def answer_pipeline(user_query: str, reference_corpus: list[str], model_name: str="gpt-4o-mini"):
    trace_id=str(uuid.uuid4())[:8]
    user_query=_normalize(user_query)

    # 1) Hardcoded input restriction
    ok,reason=is_query_allowed(user_query)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 2) Init LLM
    try: llm=get_llm(model_name)
    except Exception as e: return {"status":"config_error","reason":str(e),"trace":trace_id}

    # 3) Hardcoded malicious code check
    code_block=_extract_code_block(user_query)
    ok,reason=validate_user_query(user_query,code_block)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 4) LLM safety filter
    ok,reason=validate_with_llm(llm,user_query)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 5) Invoke
    try: resp=safe_invoke(llm,user_query)
    except Exception as e: return {"status":"llm_error","reason":str(e),"trace":trace_id}

    content, trunc=_truncate(resp["content"],MAX_OUTPUT_CHARS)

    # 6) Plagiarism
    check=check_plagiarism(content,reference_corpus)
    if check["flagged"]:
        return {"status":"plagiarism_flagged","details":check,"llm_preview":content[:200],"trace":trace_id}

    return {"status":"ok","answer":content,"trace":trace_id}

Overwriting app_mcp.py


In [15]:
REFERENCE_CORPUS = [
    "This is an example paper text ...",
    "Another doc with content ..."
]

result = answer_pipeline("Explain quantum computing basics", REFERENCE_CORPUS)
print(result)

{'status': 'blocked', 'reason': 'Blocked by LLM safety filter', 'trace': '2cdb1b97'}


In [19]:
%%writefile app_mcp_demo.py
import os, uuid, re, ast, unicodedata
from collections import Counter
from math import sqrt
import gradio as gr

# -----------------------------
# CONFIG
# -----------------------------
ALLOWED_MODELS = {"gpt-4o-mini", "gpt-4o"}
MAX_OUTPUT_CHARS = 4000

# -----------------------------
# HELPERS
# -----------------------------
def _normalize(text: str) -> str:
    return unicodedata.normalize("NFKC", text).strip()

def _truncate(s: str, limit: int) -> tuple[str, bool]:
    return (s[:limit], len(s) > limit)

def _extract_code_block(query: str) -> str | None:
    start = query.find("```")
    if start == -1: return None
    end = query.find("```", start + 3)
    if end == -1: return None
    return query[start + 3:end].strip()

# -----------------------------
# INPUT SAFETY (hardcoded)
# -----------------------------
SENSITIVE_CATEGORIES = {
    "self_harm": [r"\bsuicide\b", r"\bkill myself\b", r"\bself harm\b"],
    "violence": [r"\bkill\b", r"\bmake a bomb\b", r"\battack\b"],
    "illegal": [r"\bhack\b", r"\bcredit card\b", r"\bwifi password\b", r"\bcrack\b"],
    "personal_data": [r"\bssn\b", r"\baadhaar\b", r"\bpan\b", r"\bupi\b", r"\bpassword\b"],
}

def is_query_allowed(text: str, max_len: int = 5000) -> tuple[bool, str]:
    if not text or len(text.strip()) < 3:
        return False, "Empty/too short query."
    if len(text) > max_len:
        return False, f"Query too long ({len(text)} chars). Limit: {max_len}."
    lower = text.lower()
    for cat, patterns in SENSITIVE_CATEGORIES.items():
        for p in patterns:
            if re.search(p, lower):
                return False, f"Blocked sensitive content: {cat}"
    return True, "OK"

# -----------------------------
# MALICIOUS CODE CHECK (hardcoded)
# -----------------------------
DANGEROUS_PATTERNS = [
    r"\bos\.system\s*\(",
    r"\bsubprocess\.(Popen|run|call)\s*\(",
    r"\beval\s*\(",
    r"\bexec\s*\(",
    r"\brm\s+-rf\b",
]

FORBIDDEN_CALL_NAMES = {"eval","exec","compile","os.system","subprocess.Popen","subprocess.call","subprocess.run"}

def contains_dangerous_strings(s: str) -> list[str]:
    s_lower = s.lower()
    return [pat for pat in DANGEROUS_PATTERNS if re.search(p, s_lower)]

def validate_user_query(query: str, code_snippet: str | None = None) -> tuple[bool, str]:
    hits = contains_dangerous_strings(query)
    if hits:
        return False, f"Blocked due to dangerous patterns: {hits}"
    if code_snippet:
        try:
            tree = ast.parse(code_snippet)
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_CALL_NAMES:
                        return False, f"Forbidden call: {node.func.id}"
        except SyntaxError:
            return False, "Syntax error in code block."
    return True, "OK"

# -----------------------------
# PLAGIARISM CHECK
# -----------------------------
def _normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return s.strip()

def cosine_token(a: str, b: str) -> float:
    A = Counter(_normalize_text(a).split())
    B = Counter(_normalize_text(b).split())
    dot = sum(A[t]*B[t] for t in set(A)|set(B))
    magA = sqrt(sum(v*v for v in A.values()))
    magB = sqrt(sum(v*v for v in B.values()))
    return dot/(magA*magB+1e-9)

def check_plagiarism(generated: str, corpus: list[str], cosine_thresh: float=0.85) -> dict:
    best = {"source_idx":None,"cosine":0.0}
    for i,ref in enumerate(corpus):
        cos = cosine_token(generated,ref)
        if cos>best["cosine"]:
            best={"source_idx":i,"cosine":cos}
    flagged = best["cosine"]>=cosine_thresh
    return {"flagged":flagged,"best_match":best}

# -----------------------------
# LLM WRAPPER (dummy for demo)
# -----------------------------
class DummyLLM:
    def __init__(self,name,key): self.name=name; self.key=key
    def invoke(self,prompt): return {"content":f"Echo from {self.name}: {prompt[:200]}"}

def get_llm(model_name: str):
    if model_name not in ALLOWED_MODELS:
        raise ValueError(f"Unsupported model {model_name}")
    return DummyLLM(model_name,"dummy_key")

def safe_invoke(llm,prompt:str)->dict:
    resp=llm.invoke(prompt)
    if not resp or "content" not in resp: raise ValueError("Empty response")
    return resp

# -----------------------------
# LLM SAFETY FILTER (intelligent)
# -----------------------------
def llm_safety_filter(llm, query: str) -> bool:
    prompt = f"""
    You are a safety filter. Classify the following query:
    "{query}"
    Respond with only one word: SAFE or UNSAFE.
    """
    resp = llm.invoke(prompt)
    label = resp["content"].strip().upper()
    return label == "SAFE"

def validate_with_llm(llm, query: str) -> tuple[bool, str]:
    if not llm_safety_filter(llm, query):
        return False, "Blocked by LLM safety filter"
    return True, "OK"

# -----------------------------
# HYBRID PIPELINE
# -----------------------------
def answer_pipeline(user_query: str, reference_corpus: list[str], model_name: str="gpt-4o-mini"):
    trace_id=str(uuid.uuid4())[:8]
    user_query=_normalize(user_query)

    # 1) Hardcoded input restriction
    ok,reason=is_query_allowed(user_query)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 2) Init LLM
    try: llm=get_llm(model_name)
    except Exception as e: return {"status":"config_error","reason":str(e),"trace":trace_id}

    # 3) Hardcoded malicious code check
    code_block=_extract_code_block(user_query)
    ok,reason=validate_user_query(user_query,code_block)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 4) LLM safety filter
    ok,reason=validate_with_llm(llm,user_query)
    if not ok: return {"status":"blocked","reason":reason,"trace":trace_id}

    # 5) Invoke
    try: resp=safe_invoke(llm,user_query)
    except Exception as e: return {"status":"llm_error","reason":str(e),"trace":trace_id}

    content, trunc=_truncate(resp["content"],MAX_OUTPUT_CHARS)

    # 6) Plagiarism
    check=check_plagiarism(content,reference_corpus)
    if check["flagged"]:
        return {"status":"plagiarism_flagged","details":check,"llm_preview":content[:200],"trace":trace_id}

    return {"status":"ok","answer":content,"trace":trace_id}

# -----------------------------
# GRADIO DEMO
# -----------------------------
REFERENCE_CORPUS = [
    "This is an example paper text ...",
    "Another doc with content ..."
]

def run_agent(user_query):
    result = answer_pipeline(user_query, REFERENCE_CORPUS, model_name="gpt-4o-mini")
    return result  # return full JSON so judges see metadata

demo = gr.Interface(
    fn=run_agent,
    inputs=gr.Textbox(lines=4, placeholder="Enter your query here..."),
    outputs="json",
    title="🛡️ Hybrid Failsafe Agent MCP",
    description="Demonstrates hardcoded + LLM safety checks, plagiarism detection, and safe invocation."
)

if __name__ == "__main__":
    # Reference corpus for plagiarism check
    REFERENCE_CORPUS = [
        "This is an example paper text ...",
        "Another doc with content ..."
    ]

    # Example queries to demonstrate each guardrail
    test_queries = {
        "Safe Query": "Explain quantum computing basics",
        "Sensitive Query": "I want to kill myself",
        "Malicious Code": "import os\nos.system('rm -rf /')",
        "Plagiarism": "Echo from gpt-4o-mini: This is an example paper text ...",
        "Oversized Input": "A" * 6000
    }

    print("\n=== Failsafe Agent Demo ===\n")
    for label, query in test_queries.items():
        print(f"--- {label} ---")
        result = answer_pipeline(query, REFERENCE_CORPUS)
        # Pretty-print each category outcome
        for check, outcome in result["checks"].items():
            print(f"{check}: {outcome['status']} ({outcome['reason']})")
        print(f"Final Outcome: {result['final']}")
        print(f"Trace ID: {result['trace']}\n")

Overwriting app_mcp_demo.py


In [20]:
!pip install gradio --quiet
!python app_mcp_demo.py


=== Failsafe Agent Demo ===

--- Safe Query ---
Traceback (most recent call last):
  File "/kaggle/working/app_mcp_demo.py", line 219, in <module>
    result = answer_pipeline(query, REFERENCE_CORPUS)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/app_mcp_demo.py", line 160, in answer_pipeline
    ok,reason=validate_user_query(user_query,code_block)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/app_mcp_demo.py", line 68, in validate_user_query
    hits = contains_dangerous_strings(query)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/app_mcp_demo.py", line 65, in contains_dangerous_strings
    return [pat for pat in DANGEROUS_PATTERNS if re.search(p, s_lower)]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/app_mcp_demo.py", line 65, in <listcomp>
    return [pat for pat in DANGEROUS_PATTERNS if re.search(p, s_lower)]
                                 

In [21]:
%%writefile failsafe_agent.py
import os, uuid, re, ast, unicodedata
from collections import Counter
from math import sqrt

ALLOWED_MODELS = {"gpt-4o-mini", "gpt-4o"}
MAX_OUTPUT_CHARS = 4000

# -----------------------------
# Helpers
# -----------------------------
def _normalize(text: str) -> str:
    return unicodedata.normalize("NFKC", text).strip()

def _truncate(s: str, limit: int) -> tuple[str, bool]:
    return (s[:limit], len(s) > limit)

def _extract_code_block(query: str) -> str | None:
    start = query.find("```")
    if start == -1: return None
    end = query.find("```", start + 3)
    if end == -1: return None
    return query[start + 3:end].strip()

# -----------------------------
# Input Restriction
# -----------------------------
SENSITIVE_CATEGORIES = {
    "self_harm": [r"\bsuicide\b", r"\bkill myself\b", r"\bself harm\b"],
    "violence": [r"\bkill\b", r"\bmake a bomb\b", r"\battack\b"],
    "illegal": [r"\bhack\b", r"\bcredit card\b", r"\bwifi password\b", r"\bcrack\b"],
    "personal_data": [r"\bssn\b", r"\baadhaar\b", r"\bpan\b", r"\bupi\b", r"\bpassword\b"],
}

def input_restriction_check(text: str, max_len: int = 5000) -> tuple[str,str]:
    if not text or len(text.strip()) < 3:
        return "blocked", "Empty/too short query"
    if len(text) > max_len:
        return "blocked", f"Query too long ({len(text)} chars). Limit: {max_len}"
    lower = text.lower()
    for cat, patterns in SENSITIVE_CATEGORIES.items():
        for p in patterns:
            if re.search(p, lower):
                return "blocked", f"Sensitive content: {cat}"
    return "passed", "OK"

# -----------------------------
# Malicious Code Check
# -----------------------------
DANGEROUS_PATTERNS = [
    r"\bos\.system\s*\(",
    r"\bsubprocess\.(Popen|run|call)\s*\(",
    r"\beval\s*\(",
    r"\bexec\s*\(",
    r"\brm\s+-rf\b",
]

FORBIDDEN_CALL_NAMES = {"eval","exec","compile","os.system","subprocess.Popen","subprocess.call","subprocess.run"}

def contains_dangerous_strings(s: str) -> list[str]:
    s_lower = s.lower()
    return [pat for pat in DANGEROUS_PATTERNS if re.search(pat, s_lower)]

def malicious_code_check(query: str, code_snippet: str | None = None) -> tuple[str,str]:
    hits = contains_dangerous_strings(query)
    if hits:
        return "blocked", f"Dangerous patterns: {hits}"
    if code_snippet:
        try:
            tree = ast.parse(code_snippet)
            for node in ast.walk(tree):
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_CALL_NAMES:
                        return "blocked", f"Forbidden call: {node.func.id}"
        except SyntaxError:
            return "blocked", "Syntax error in code block"
    return "passed", "OK"

# -----------------------------
# Plagiarism Check
# -----------------------------
def _normalize_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    return s.strip()

def cosine_token(a: str, b: str) -> float:
    A = Counter(_normalize_text(a).split())
    B = Counter(_normalize_text(b).split())
    dot = sum(A[t]*B[t] for t in set(A)|set(B))
    magA = sqrt(sum(v*v for v in A.values()))
    magB = sqrt(sum(v*v for v in B.values()))
    return dot/(magA*magB+1e-9)

def plagiarism_check(generated: str, corpus: list[str], cosine_thresh: float=0.85) -> tuple[str,str]:
    best = {"source_idx":None,"cosine":0.0}
    for i,ref in enumerate(corpus):
        cos = cosine_token(generated,ref)
        if cos>best["cosine"]:
            best={"source_idx":i,"cosine":cos}
    if best["cosine"]>=cosine_thresh:
        return "flagged", f"Similarity {best['cosine']:.2f} with corpus[{best['source_idx']}]"
    return "passed", "OK"

# -----------------------------
# Dummy LLM + Safety Filter
# -----------------------------
class DummyLLM:
    def __init__(self,name,key): self.name=name; self.key=key
    def invoke(self,prompt): return {"content":f"Echo from {self.name}: {prompt[:200]}"}

def llm_safety_filter(llm, query: str) -> tuple[str,str]:
    if "hack" in query.lower() or "rm -rf" in query.lower():
        return "blocked", "LLM safety filter flagged"
    return "passed", "OK"

# -----------------------------
# Pipeline
# -----------------------------
def answer_pipeline(user_query: str, reference_corpus: list[str], model_name: str="gpt-4o-mini"):
    trace_id=str(uuid.uuid4())[:8]
    user_query=_normalize(user_query)
    results = {}

    # Input restriction
    status,reason = input_restriction_check(user_query)
    results["Input Restriction"] = {"status":status,"reason":reason}
    if status=="blocked": return {"trace":trace_id,"checks":results,"final":"blocked"}

    # Malicious code
    code_block=_extract_code_block(user_query)
    status,reason = malicious_code_check(user_query,code_block)
    results["Malicious Code"] = {"status":status,"reason":reason}
    if status=="blocked": return {"trace":trace_id,"checks":results,"final":"blocked"}

    # LLM safety filter
    llm=DummyLLM(model_name,"dummy_key")
    status,reason = llm_safety_filter(llm,user_query)
    results["LLM Safety Filter"] = {"status":status,"reason":reason}
    if status=="blocked": return {"trace":trace_id,"checks":results,"final":"blocked"}

    # Invoke dummy LLM
    resp=llm.invoke(user_query)
    content,_=_truncate(resp["content"],MAX_OUTPUT_CHARS)

    # Plagiarism
    status,reason = plagiarism_check(content,reference_corpus)
    results["Plagiarism"] = {"status":status,"reason":reason}
    if status=="flagged": return {"trace":trace_id,"checks":results,"final":"plagiarism_flagged"}

    results["Answer"] = {"status":"ok","content":content}
    return {"trace":trace_id,"checks":results,"final":"ok"}

# -----------------------------
# Demo Runner (Markdown tables)
# -----------------------------
if __name__ == "__main__":
    REFERENCE_CORPUS = [
        "This is an example paper text ...",
        "Another doc with content ..."
    ]

    test_queries = {
        "Safe Query": "Explain quantum computing basics",
        "Sensitive Query": "I want to kill myself",
        "Malicious Code": "import os\nos.system('rm -rf /')",
        "Plagiarism": "Echo from gpt-4o-mini: This is an example paper text ...",
        "Oversized Input": "A" * 6000
    }

    print("\n=== Failsafe Agent Demo ===\n")
    for label, query in test_queries.items():
        result = answer_pipeline(query, REFERENCE_CORPUS)
        print(f"--- {label} ---")
        print(f"Trace ID: {result['trace']}")
        print("| Check Category       | Status   | Reason/Content |")
        print("|----------------------|----------|----------------|")
        for check, outcome in result["checks"].items():
            reason = outcome.get("reason","")
            if check=="Answer":
                reason = outcome.get("content","")
            print(f"| {check:<20} | {outcome['status']:<8} | {reason[:60]} |")
        print(f"Final Outcome: {result['final']}\n")

Overwriting failsafe_agent.py


In [22]:
!python failsafe_agent.py


=== Failsafe Agent Demo ===

--- Safe Query ---
Trace ID: 8aeaf7ef
| Check Category       | Status   | Reason/Content |
|----------------------|----------|----------------|
| Input Restriction    | passed   | OK |
| Malicious Code       | passed   | OK |
| LLM Safety Filter    | passed   | OK |
| Plagiarism           | passed   | OK |
| Answer               | ok       | Echo from gpt-4o-mini: Explain quantum computing basics |
Final Outcome: ok

--- Sensitive Query ---
Trace ID: 217841f9
| Check Category       | Status   | Reason/Content |
|----------------------|----------|----------------|
| Input Restriction    | blocked  | Sensitive content: self_harm |
Final Outcome: blocked

--- Malicious Code ---
Trace ID: 01b37c7f
| Check Category       | Status   | Reason/Content |
|----------------------|----------|----------------|
| Input Restriction    | passed   | OK |
| Malicious Code       | blocked  | Dangerous patterns: ['\\bos\\.system\\s*\\(', '\\brm\\s+-rf\ |
Final Outcome: blocke